<a href="https://colab.research.google.com/github/Deepikadandolu/SumInt_IISc_ex1/blob/main/Tumour_transfer_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import numpy as np
import os
import cv2
import matplotlib.pyplot as plt
import kagglehub
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Dense,
    Dropout,
    GlobalAveragePooling2D)
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.applications import MobileNetV2

In [5]:
#path
path = kagglehub.dataset_download(
    "sartajbhuvaji/brain-tumor-classification-mri")
print(path)

Using Colab cache for faster access to the 'brain-tumor-classification-mri' dataset.
/kaggle/input/brain-tumor-classification-mri


In [6]:
# dataset loading
classes = {
    'no_tumor': 0,
    'pituitary_tumor': 1,
    'glioma_tumor': 2,
    'meningioma_tumor': 3}
train_path = os.path.join(path, "Training")
X = []
Y = []

for cls, label in classes.items():
    class_path = os.path.join(train_path, cls)
    for filename in os.listdir(class_path):
        img_path = os.path.join(class_path, filename)
        img = cv2.imread(img_path)
        if img is not None:
            img = cv2.cvtColor(
                img,
                cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (224, 224))
            img = img / 255.0
            X.append(img)
            Y.append(label)
X = np.array(X)
Y = np.array(Y)
Y = to_categorical(Y, num_classes=4)
print(X.shape)
print(Y.shape)

(2870, 224, 224, 3)
(2870, 4)


In [4]:
#train_test split
xtrain, xtest, ytrain, ytest = train_test_split(
    X,
    Y,
    test_size=0.2,
    random_state=10,
    stratify=Y
)

In [7]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    GlobalAveragePooling2D,
    Dense,
    Dropout
)

In [8]:
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)
base_model.trainable = False
model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(4, activation='softmax')
])
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,468 (9.24 MB)

 Trainable params: 164,484 (642.52 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [11]:
#train
from sklearn.model_selection import train_test_split
xtrain, xtest, ytrain, ytest = train_test_split(
    X,
    Y,
    test_size=0.2,
    random_state=10,
    stratify=Y
)
print(xtrain.shape)
print(xtest.shape)

history = model.fit(
    xtrain,
    ytrain,
    validation_data=(xtest, ytest),
    epochs=5,
    batch_size=32
)

(2296, 224, 224, 3)
(574, 224, 224, 3)
Epoch 1/5
72/72 ━━━━━━━━━━━━━━━━━━━━ 93s 1s/step - accuracy: 0.6890 - loss: 0.7759 - val_accuracy: 0.8310 - val_loss: 0.4398
Epoch 2/5
72/72 ━━━━━━━━━━━━━━━━━━━━ 82s 1s/step - accuracy: 0.8153 - loss: 0.4915 - val_accuracy: 0.8589 - val_loss: 0.3763
Epoch 3/5
72/72 ━━━━━━━━━━━━━━━━━━━━ 84s 1s/step - accuracy: 0.8367 - loss: 0.4046 - val_accuracy: 0.8815 - val_loss: 0.3134
Epoch 4/5
72/72 ━━━━━━━━━━━━━━━━━━━━ 72s 1s/step - accuracy: 0.8571 - loss: 0.3559 - val_accuracy: 0.8763 - val_loss: 0.3346
Epoch 5/5
72/72 ━━━━━━━━━━━━━━━━━━━━ 90s 1s/step - accuracy: 0.8824 - loss: 0.3066 - val_accuracy: 0.8763 - val_loss: 0.3317


In [12]:
#test
test_loss, test_accuracy = model.evaluate(
    xtest,
    ytest
)

print("Test Accuracy:", test_accuracy)

18/18 ━━━━━━━━━━━━━━━━━━━━ 14s 757ms/step - accuracy: 0.8763 - loss: 0.3317
Test Accuracy: 0.8763065934181213


In [13]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Predictions
ypred = model.predict(xtest)

# Convert one-hot vectors to class labels
ytest_labels = np.argmax(ytest, axis=1)
ypred_labels = np.argmax(ypred, axis=1)

# Training Accuracy
train_loss, train_accuracy = model.evaluate(
    xtrain,
    ytrain,
    verbose=0
)

# Testing Accuracy
test_loss, test_accuracy = model.evaluate(
    xtest,
    ytest,
    verbose=0
)

# F1 Score
f1 = f1_score(
    ytest_labels,
    ypred_labels,
    average='weighted'
)

# Confusion Matrix
cm = confusion_matrix(
    ytest_labels,
    ypred_labels
)

# Sensitivity
sensitivity = np.mean(
    cm.diagonal() / cm.sum(axis=1)
)

# Specificity
specificity = np.mean([
    (np.sum(cm) - (
        cm[i,:].sum() +
        cm[:,i].sum() -
        cm[i,i]
    )) /
    (np.sum(cm) - cm[i,:].sum())
    for i in range(len(cm))
])

# Print Results
print("Training Accuracy:", train_accuracy)
print("Testing Accuracy:", test_accuracy)
print("F1-Score:", f1)
print("Sensitivity:", sensitivity)
print("Specificity:", specificity)

# Classification Report
print("\nClassification Report:\n")
print(
    classification_report(
        ytest_labels,
        ypred_labels
    )
)
# Confusion Matrix
print("\nConfusion Matrix:\n")
print(cm)

18/18 ━━━━━━━━━━━━━━━━━━━━ 16s 835ms/step
Training Accuracy: 0.9224738478660583
Testing Accuracy: 0.8763065934181213
F1-Score: 0.875891654982693
Sensitivity: 0.86253970666691
Specificity: 0.957063458130005

Classification Report:

              precision    recall  f1-score   support

           0       0.94      0.77      0.85        79
           1       0.95      0.93      0.94       166
           2       0.87      0.94      0.90       165
           3       0.80      0.80      0.80       164

    accuracy                           0.88       574
   macro avg       0.89      0.86      0.87       574
weighted avg       0.88      0.88      0.88       574


Confusion Matrix:

[[ 61   2   2  14]
 [  0 155   1  10]
 [  0   0 155  10]
 [  4   7  21 132]]


In [14]:
model.save("brain_tumor_transfer_learning.keras")